In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [2]:
data = load_diabetes()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [3]:
X.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641


In [4]:
y[:5]

array([151.,  75., 141., 206., 135.])

In [ ]:
y = (y > np.median(y)).astype(int)

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [6]:
baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

In [7]:
print(X_train.head())

          age       sex       bmi        bp        s1        s2        s3  \
17   0.070769  0.050680  0.012117  0.056301  0.034206  0.049416 -0.039719   
66  -0.009147  0.050680 -0.018062 -0.033213 -0.020832  0.012152 -0.072854   
137  0.005383 -0.044642  0.049840  0.097615 -0.015328 -0.016345 -0.006584   
245 -0.027310 -0.044642 -0.035307 -0.029770 -0.056607 -0.058620  0.030232   
31  -0.023677 -0.044642 -0.065486 -0.081413 -0.038720 -0.053610  0.059685   

           s4        s5        s6  
17   0.034309  0.027364 -0.001078  
66   0.071210  0.000272  0.019633  
137 -0.002592  0.017036 -0.013504  
245 -0.039493 -0.049872 -0.129483  
31  -0.076395 -0.037129 -0.042499  


In [8]:
print("Baseline Accuracy:", accuracy_score(y_test, baseline_pred))

Baseline Accuracy: 0.0


In [9]:
ew = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')

X_train_ew = ew.fit_transform(X_train)
X_test_ew = ew.transform(X_test)

In [10]:
df_ew = pd.DataFrame(X_train_ew, columns=X.columns)
print(df_ew.head())

   age  sex  bmi   bp   s1   s2   s3   s4   s5   s6
0  4.0  4.0  2.0  3.0  2.0  2.0  1.0  2.0  2.0  2.0
1  2.0  4.0  1.0  1.0  1.0  2.0  0.0  2.0  2.0  2.0
2  2.0  0.0  2.0  4.0  1.0  1.0  1.0  1.0  2.0  2.0
3  1.0  0.0  1.0  1.0  0.0  0.0  2.0  0.0  1.0  0.0
4  1.0  0.0  0.0  0.0  1.0  0.0  2.0  0.0  1.0  1.0


In [11]:
model_ew = LogisticRegression(max_iter=1000)
model_ew.fit(X_train_ew, y_train)

ew_pred = model_ew.predict(X_test_ew)
print("Equal Width Accuracy:", accuracy_score(y_test, ew_pred))

Equal Width Accuracy: 0.0


In [12]:
ef = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

X_train_ef = ef.fit_transform(X_train)
X_test_ef = ef.transform(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_discretization.py:306: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 1 are removed. Consider decreasing the number of bins.
  warnings.warn(


In [13]:
df_ef = pd.DataFrame(X_train_ef, columns=X.columns)
print(df_ef.head())

   age  sex  bmi   bp   s1   s2   s3   s4   s5   s6
0  4.0  0.0  3.0  4.0  3.0  4.0  1.0  4.0  3.0  2.0
1  1.0  0.0  1.0  1.0  1.0  3.0  0.0  4.0  2.0  3.0
2  2.0  0.0  4.0  4.0  1.0  1.0  2.0  3.0  3.0  1.0
3  1.0  0.0  1.0  1.0  0.0  0.0  3.0  1.0  0.0  0.0
4  1.0  0.0  0.0  0.0  1.0  0.0  4.0  0.0  1.0  0.0


In [14]:
model_ef = LogisticRegression(max_iter=1000)
model_ef.fit(X_train_ef, y_train)

ef_pred = model_ef.predict(X_test_ef)
print("Equal Frequency Accuracy:", accuracy_score(y_test, ef_pred))

Equal Frequency Accuracy: 0.0


In [15]:
X_train_km = X_train.copy()
X_test_km = X_test.copy()

for col in X.columns:
    kmeans = KMeans(n_clusters=5, random_state=42)

    X_train_km[col] = kmeans.fit_predict(X_train[[col]])
    X_test_km[col] = kmeans.predict(X_test[[col]])

df_km = X_train_km.copy()
print(df_km.head())

     age  sex  bmi  bp  s1  s2  s3  s4  s5  s6
17     1    1    2   1   0   0   0   0   0   1
66     0    1    1   0   3   4   4   4   0   3
137    2    0    3   3   3   1   2   2   0   1
245    0    0    1   0   1   3   1   1   1   2
31     0    0    4   2   3   3   1   1   4   4


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (2) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [16]:
model_km = LogisticRegression(max_iter=1000)
model_km.fit(X_train_km, y_train)

km_pred = model_km.predict(X_test_km)
print("K-Means Accuracy:", accuracy_score(y_test, km_pred))

K-Means Accuracy: 0.02247191011235955


In [17]:
from sklearn.preprocessing import Binarizer

In [19]:
threshold = 0.0
binarizer = Binarizer(threshold=threshold)

In [20]:
X_train_bin = binarizer.fit_transform(X_train)
X_test_bin = binarizer.transform(X_test)

In [21]:
df_bin = pd.DataFrame(X_train_bin, columns=X.columns)
print(df_bin.head())

   age  sex  bmi   bp   s1   s2   s3   s4   s5   s6
0  1.0  1.0  1.0  1.0  1.0  1.0  0.0  1.0  1.0  0.0
1  0.0  1.0  0.0  0.0  0.0  1.0  0.0  1.0  1.0  1.0
2  1.0  0.0  1.0  1.0  0.0  0.0  0.0  0.0  1.0  0.0
3  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0
4  0.0  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0


In [22]:
model_bin = LogisticRegression(max_iter=1000)
model_bin.fit(X_train_bin, y_train)

bin_pred = model_bin.predict(X_test_bin)
bin_acc = accuracy_score(y_test, bin_pred)

print("Binarization Accuracy:", bin_acc)

Binarization Accuracy: 0.011235955056179775
